In [2]:

from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from tslearn.shapelets import LearningShapelets
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import json
import warnings
import stumpy
import os
warnings.filterwarnings('ignore')

np.random.seed(42)

df_ts = pd.read_csv("../timeseries_datasets/tracks_timeseries.csv")

artist_name_map = {
    "ART25707984": "fabri fibra",
    "ART07024718": "fedez"
}

2025-12-28 21:43:36.450956: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
ERROR:numba.cuda.cudadrv.driver:Call to cuInit results in CUDA_ERROR_NO_DEVICE


Below we map artist IDs to names and convert the JSON encoded time series features into numpy arrays for processing. We display dataset statistics with song counts per artist

In [3]:

# get artist name and convert to array the timeseries rms
df_ts['artist_name'] = df_ts['id_artist'].apply(lambda x: artist_name_map.get(x))

for col in ['rms','centroid', 'rolloff', 'flux', 'zcr', 'spectral_bw']:
    df_ts[col] = df_ts[col].apply(lambda x: np.array(json.loads(x)) if isinstance(x, str) else x)

print(f"Fedez songs counts: {len(df_ts[df_ts['artist_name'] == 'fedez'])}")
print(f"Fabri Fibra songs counts: {len(df_ts[df_ts['artist_name'] == 'fabri fibra'])}")

Fedez songs counts: 171
Fabri Fibra songs counts: 269


Scale timeseries and extract a fixed number of samples (`target_ts_len`) to reduce timeseries length to speed up processing while preserving signal characteristics

In [4]:
target_ts_len = 200

stacked = np.vstack(df_ts['rms'].values)[:, :target_ts_len]
scaler = StandardScaler()
df_ts['rms_scaled'] = list(scaler.fit_transform(stacked))

We extract a sample of 30 songs per artist for initial motif and anomaly exploration. Using sampling, ensures both artists are equally represented in the analysis.

In [5]:
sample_size = 30
fedez_idx = df_ts[df_ts['id_artist'] == 'ART07024718'].sample(sample_size, random_state=42).index
fibra_idx = df_ts[df_ts['id_artist'] == 'ART25707984'].sample(sample_size, random_state=42).index
df_ts_sample = df_ts.loc[list(fedez_idx) + list(fibra_idx)].copy().reset_index(drop=True)

print(f"Total songs to analyze: {len(df_ts_sample)}. {sample_size} per artist")

Total songs to analyze: 60. 30 per artist


Functions to find motifs and anomalies
exclusion zone is used to prevent overlapping motifs extraction. Both functions use `STUMPY` matrix profile algorithm with an exclusion zone (window_size/2) to prevent overlapping detections.

In [6]:
def find_motifs(data, window_size, n_motifs=3):
    mp = stumpy.stump(data, m=window_size)
    mp_values = mp[:, 0].copy()
    exclusion_zone = window_size // 2
    
    motifs = []
    for i in range(n_motifs):
        motif_idx = np.argmin(mp_values)
        neighbor_idx = int(mp[motif_idx, 1])
        dist = mp[motif_idx, 0]
        motifs.append((motif_idx, neighbor_idx, dist))
        
        for idx in [motif_idx, neighbor_idx]:
            start = max(0, idx - exclusion_zone)
            end = min(len(mp_values), idx + exclusion_zone + 1)
            mp_values[start:end] = np.inf
    
    return {'matrix_profile': mp, 'motifs': motifs}


def find_anomalies(data, window_size, n_anomalies=3):
    mp = stumpy.stump(data, m=window_size)
    mp_values = mp[:, 0].copy()
    exclusion_zone = window_size // 2
    
    anomalies = []
    for i in range(n_anomalies):
        idx = np.argmax(mp_values)
        dist = mp[idx, 0]
        anomalies.append((idx, dist))
        
        start = max(0, idx - exclusion_zone)
        end = min(len(mp_values), idx + exclusion_zone + 1)
        mp_values[start:end] = -np.inf
    
    return {'matrix_profile': mp, 'anomalies': anomalies}

Utility functions to plot motifs and anomalies. We used interactive Plotly visualizations showing:
- Top: time series with highlighted motif and anomaly regions
- Bottom: matrix profile values

Different colors distinguish between different motifs (or anomalies) in the same series.

In [7]:

def plot_motifs(data, mp, motifs, window_size, title='Motifs'):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=['RMS Timeserie', 'Matrix Profile'])
    
    fig.add_trace(go.Scatter(y=data, mode='lines'), row=1, col=1)
    
    colors = ['rgba(99,110,250,0.4)', 'rgba(0,204,150,0.4)', 'rgba(239,85,59,0.4)']
    for i, (motif_idx, neighbor_idx, dist) in enumerate(motifs):
        for idx in [motif_idx, neighbor_idx]:
            fig.add_vrect(x0=idx, x1=idx + window_size, fillcolor=colors[i % 3], 
                          opacity=0.5, line_width=0, row=1, col=1)
            fig.add_vline(x=idx, line_dash='dash', line_color='grey', row=2, col=1)
    
    fig.add_trace(go.Scatter(y=mp[:, 0], mode='lines'), row=2, col=1)
    
    fig.update_layout(title_text=title, height=600, showlegend=False)
    fig.update_yaxes(title_text='Value', row=1, col=1)
    fig.update_yaxes(title_text='Matrix Profile', row=2, col=1)
    return fig


def plot_anomalies(data, mp, anomalies, window_size, title='Anomaly Discovery'):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=['RMS Timeserie', 'Matrix Profile'])
    
    fig.add_trace(go.Scatter(y=data, mode='lines'), row=1, col=1)
    
    colors = ['rgba(255,0,0,0.4)', 'rgba(255,99,71,0.4)', 'rgba(255,165,0,0.4)']
    for i, (idx, dist) in enumerate(anomalies):
        fig.add_vrect(x0=idx, x1=idx + window_size, fillcolor=colors[i % 3],
                      opacity=0.5, line_width=0, row=1, col=1)
        fig.add_vline(x=idx, line_dash='dash', line_color='red', row=2, col=1)
    
    fig.add_trace(go.Scatter(y=mp[:, 0], mode='lines', name='Matrix Profile'), row=2, col=1)
    
    fig.update_layout(title_text=title, height=600, showlegend=False)
    fig.update_yaxes(title_text='Value', row=1, col=1)
    fig.update_yaxes(title_text='Matrix Profile', row=2, col=1)
    return fig

## Fedez Motif Analysis

We analyze the first Fedez song from the sample to identify the top 3 motifs using a sliding window of 15 samples. We then visualize where motifs occur in the RMS time series.

In [8]:

window_size = 15

data_fedez = df_ts_sample[df_ts_sample['artist_name'] == 'fedez'].iloc[0]['rms_scaled']

motif_result = find_motifs(data_fedez, window_size, n_motifs=3)
fig = plot_motifs(data_fedez, motif_result['matrix_profile'], motif_result['motifs'], 
                  window_size, title='Fedez Motifs')
fig.show()

In [9]:
anomaly_result = find_anomalies(data_fedez, window_size, n_anomalies=3)
fig = plot_anomalies(data_fedez, anomaly_result['matrix_profile'], anomaly_result['anomalies'],
                     window_size, title='Fedez Anomalies')
fig.show()


# Fabri Fibra
We repeat the same process from Fedez in Fabri Fibra's songs

In [10]:
data_fibra = df_ts_sample[df_ts_sample['artist_name'] == 'fabri fibra'].iloc[0]['rms_scaled']

motif_result = find_motifs(data_fibra, window_size, n_motifs=3)
fig = plot_motifs(data_fibra, motif_result['matrix_profile'], motif_result['motifs'],
                  window_size, title='Fabri Fibra Motifs')
fig.show()


anomaly_result = find_anomalies(data_fibra, window_size, n_anomalies=3)
fig = plot_anomalies(data_fibra, anomaly_result['matrix_profile'], anomaly_result['anomalies'],
                     window_size, title='Fabri Fibra Anomalies')
fig.show()

## Full Dataset Analysis
Process all 440 songs (171 Fedez + 269 Fabri Fibra) to extract motif and anomaly statistics:
- **Motif distances**: lower values indicate stronger recurring patterns
- **Anomaly distances**: higher values indicate more extreme patterns

This creates a feature set for comparing artist characteristics based on motifs and anomalies distances

In [11]:
analysis_res = []
analyzed = 0

for idx, row in df_ts.iterrows():
    data = row['rms_scaled']
    motif_res = find_motifs(data, window_size, n_motifs=3)
    anomaly_res = find_anomalies(data, window_size, n_anomalies=3)
    
    motif_dists = [motif[2] for motif in motif_res['motifs']]
    anomaly_dists = [anomaly[1] for anomaly in anomaly_res['anomalies']]
    
    analysis_res.append({
        'artist': row['artist_name'],
        'motif_min_dist': np.min(motif_dists),
        'motif_mean_dist': np.mean(motif_dists),
        'anomaly_max_dist': np.max(anomaly_dists),
        'anomaly_mean_dist': np.mean(anomaly_dists)
    })
    analyzed += 1

print(f"Analyzed: {analyzed} songs")
analysis_res_df = pd.DataFrame(analysis_res)

Analyzed: 440 songs


## Summary 

We aggregate motif and anomaly metrics by artist to identify potential characteristics. Compare mean and standard deviation values to check whether features differ significantly between Fedez and Fabri Fibra

In [12]:
analysis_summary = analysis_res_df.groupby('artist').agg(['mean', 'std'])
analysis_summary.head()

motif_min_dist           motif_mean_dist            \
                      mean       std            mean       std   
artist                                                           
fabri fibra       0.539493  0.201031        0.632165  0.199427   
fedez             0.602510  0.249362        0.694971  0.253211   

            anomaly_max_dist           anomaly_mean_dist            
                        mean       std              mean       std  
artist                                                              
fabri fibra         3.179151  0.443683          2.858266  0.335873  
fedez               3.170098  0.424831          2.848397  0.357978

## Motif vs Anomaly Distributions

We used box plots to compare motif and anomaly distance distributions between the two artists. This visualization helps in identifying:
- if one artist has more consistent patterns, so lower motif distances
- if one artist has more anomalies, so higher anomaly distances

In [13]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['Motif Distance', 
                                                      'Anomaly Distance'])

for i, artist in enumerate(['fedez', 'fabri fibra']):
    artist_data = analysis_res_df[analysis_res_df['artist'] == artist]
    color = '#3498db' if artist == 'fedez' else '#e74c3c'
    
    fig.add_trace(go.Box(y=artist_data['motif_mean_dist'], name=artist.title(), 
                         marker_color=color, showlegend=False), row=1, col=1)
    fig.add_trace(go.Box(y=artist_data['anomaly_max_dist'], name=artist.title(),
                         marker_color=color, showlegend=False), row=1, col=2)

fig.update_layout(title_text='Artist Comparison: Motifs vs Anomalies', height=500)
fig.show()

## Shapelets

**Shapelets** are discriminative subsequences that best distinguish between time series classes.

The `find_shapelet_match` function is used to put all code in one place to:

1. Prepare and scale tim series data for the provided audio feature timeserie (`ts`)
2. Split data into train and test sets with stratification to maintain class balance
3. Use Grabocka's method to determine optimal shapelet sizes
4. Train a shapelet classifier using gradient descent
5. Evaluate classification accuracy on both training and test sets
6. Visualize the learned shapelets and their distances to each artist's songs
7. Show shapelet matches on example songs to interpret what patterns distinguish the artists

In [18]:
from tslearn.shapelets import LearningShapelets, grabocka_params_to_shapelet_size_dict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from keras.optimizers import Adam

def find_shapelet_match(ts, shapelet): 
    distance_profile = stumpy.mass(shapelet, ts)
    best_pos = np.argmin(distance_profile)
    min_dist = distance_profile[best_pos] 
    return best_pos, min_dist

def find_shapelet(feature_column):
    # scale data
    X_original = np.vstack(df_ts[feature_column].values)[:, :target_ts_len]

    scaler = StandardScaler()
    X = scaler.fit_transform(X_original)

    y = np.array([0 if a == 'fedez' else 1 for a in df_ts['artist_name']])

    print(f"X: {X.shape} y: {y.shape}")
    print(f"Target counts. Fedez: {sum(y==0)} Fabri Fibra: {sum(y==1)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=0.2,
        random_state=42,    
        stratify=y # maintains class balance in split
    )
    
    # set shapelet sizes using Grabocka
    n_ts, ts_sz = X_train.shape
    n_classes = len(set(y_train))
    
    shapelet_sizes = grabocka_params_to_shapelet_size_dict(
        n_ts=n_ts,
        ts_sz=ts_sz,
        n_classes=n_classes,
        l=0.1,
        r=1
    )

    # train classifier
    shp_clf = LearningShapelets(
        n_shapelets_per_size=shapelet_sizes,
        optimizer=Adam(0.01),
        batch_size=16,
        weight_regularizer=0.01,
        max_iter=200,
        random_state=42,
        verbose=0
    )
    shp_clf.fit(X_train, y_train)
    train_accuracy = shp_clf.score(X_train, y_train)*100
    test_accuracy = shp_clf.score(X_test, y_test)*100
    
    print(f"Training set accuracy: {train_accuracy:.2f} %")
    print(f"Test set accuracy: {test_accuracy:.2f} %")

    # plot shapelet standardized scale
    n_shapelets = min(3, len(shp_clf.shapelets_))
    fig = go.Figure()
    
    for i in range(n_shapelets):
        shapelet = shp_clf.shapelets_[i].ravel()
        fig.add_trace(go.Scatter(y=shapelet, mode='lines', name=f'Shapelet {i+1}'))
    
    fig.update_layout(title=f'Shapelets (Standardized Space) for feature: {feature_column}',
                      xaxis_title='Time', yaxis_title='Shapelet Value')
    fig.show()

    # distance to shapelets boxplot
    n_cols = 3
    n_rows = (n_shapelets + n_cols - 1) // n_cols
    fig = make_subplots(rows=n_rows, cols=n_cols, 
                        subplot_titles=[f'Shapelet num {i+1}' for i in range(n_shapelets)])
    
    for shapelet_index in range(n_shapelets):
        shapelet = shp_clf.shapelets_[shapelet_index].ravel()
        distances = {'fedez': [], 'fabri fibra': []}

        for idx, (ts_scaled, artist) in enumerate(zip(X, df_ts['artist_name'])):
            best_position, dist = find_shapelet_match(ts_scaled, shapelet)
            distances[artist].append(dist)

        row_num = shapelet_index // n_cols + 1
        col_num = shapelet_index % n_cols + 1

        fig.add_trace(go.Box(y=distances['fedez'], name='Fedez', 
                             marker_color='#3498db', showlegend=(shapelet_index==0)),
                      row=row_num, col=col_num)
        fig.add_trace(go.Box(y=distances['fabri fibra'], name='Fabri Fibra', 
                             marker_color='#e74c3c', showlegend=(shapelet_index==0)),
                      row=row_num, col=col_num)

        print(f"Shapelet number: {shapelet_index+1}. Fedez distance: {np.mean(distances['fedez']):.2f} Fabri Fibra distance: {np.mean(distances['fabri fibra']):.2f}")

    fig.update_layout(title=f'Distance to Shapelet by Artist for feature: {feature_column}', 
                      height=300*n_rows, showlegend=True)
    fig.show()

    # plot shapelet on a song
    fig = make_subplots(rows=2, cols=1, subplot_titles=['Fedez Shapelet Match', 'Fabri Fibra Shapelet Match'])
    
    # plot first shapelet
    shapelet = shp_clf.shapelets_[0].ravel()
    
    for i, artist in enumerate(['fedez', 'fabri fibra']):
        # extract scaled timeserie for song
        artist_idx = df_ts[df_ts['artist_name'] == artist].index[0]
        ts_scaled = X[artist_idx]
        
        pos, dist = find_shapelet_match(ts_scaled, shapelet)

        fig.add_trace(go.Scatter(y=ts_scaled, mode='lines', name='Time Series',
                                line=dict(color='blue', width=1)), row=i+1, col=1)
        fig.add_trace(go.Scatter(x=list(range(pos, pos+len(shapelet))), y=shapelet,
                                mode='lines', name='Shapelet',
                                line=dict(color='red', width=3)), row=i+1, col=1)
        
        fig.update_yaxes(title_text="Standardized Value", row=i+1, col=1)

    fig.update_layout(title=f'Shapelet Match in Song (Standardized Space) for feature: {feature_column}', 
                      height=600, showlegend=True)
    fig.show()

## Available Features
Below we show all available timeseries in the dataset that can be used for shapelet analysis. Wee the execute the shapelet function so that we can see the shapelet size found for each feature timeseriesnd we use a box plot to see distance distribution to each learned shapelet for each artist

In [19]:
df_ts.columns

Index(['id', 'id_artist', 'num_samples', 'sr', 'centroid', 'rolloff', 'flux',
       'rms', 'zcr', 'spectral_bw', 'artist_name', 'rms_scaled'],
      dtype='object')

In [20]:
find_shapelet("centroid")

X: (440, 200) y: (440,)
Target counts. Fedez: 171 Fabri Fibra: 269
Training set accuracy: 70.74 %
Test set accuracy: 71.59 %


Shapelet number: 1. Fedez distance: 2.91 Fabri Fibra distance: 3.05
Shapelet number: 2. Fedez distance: 3.78 Fabri Fibra distance: 3.64
Shapelet number: 3. Fedez distance: 4.90 Fabri Fibra distance: 4.87


In [21]:
find_shapelet("rolloff")

X: (440, 200) y: (440,)
Target counts. Fedez: 171 Fabri Fibra: 269
Training set accuracy: 71.02 %
Test set accuracy: 67.05 %


Shapelet number: 1. Fedez distance: 4.16 Fabri Fibra distance: 4.20
Shapelet number: 2. Fedez distance: 4.32 Fabri Fibra distance: 4.12
Shapelet number: 3. Fedez distance: 4.57 Fabri Fibra distance: 4.53


In [22]:
find_shapelet("flux")

X: (440, 200) y: (440,)
Target counts. Fedez: 171 Fabri Fibra: 269
Training set accuracy: 72.16 %
Test set accuracy: 62.50 %


Shapelet number: 1. Fedez distance: 4.53 Fabri Fibra distance: 4.63
Shapelet number: 2. Fedez distance: 4.12 Fabri Fibra distance: 4.26
Shapelet number: 3. Fedez distance: 3.40 Fabri Fibra distance: 3.20


In [23]:
find_shapelet("rms")

X: (440, 200) y: (440,)
Target counts. Fedez: 171 Fabri Fibra: 269
Training set accuracy: 71.31 %
Test set accuracy: 56.82 %


Shapelet number: 1. Fedez distance: 4.75 Fabri Fibra distance: 4.81
Shapelet number: 2. Fedez distance: 2.52 Fabri Fibra distance: 2.38
Shapelet number: 3. Fedez distance: 3.00 Fabri Fibra distance: 2.86


In [24]:
find_shapelet("zcr")

X: (440, 200) y: (440,)
Target counts. Fedez: 171 Fabri Fibra: 269
Training set accuracy: 68.18 %
Test set accuracy: 60.23 %


Shapelet number: 1. Fedez distance: 4.83 Fabri Fibra distance: 4.84
Shapelet number: 2. Fedez distance: 2.77 Fabri Fibra distance: 2.73
Shapelet number: 3. Fedez distance: 2.57 Fabri Fibra distance: 2.60


In [25]:
find_shapelet("spectral_bw")

X: (440, 200) y: (440,)
Target counts. Fedez: 171 Fabri Fibra: 269
Training set accuracy: 70.17 %
Test set accuracy: 61.36 %


Shapelet number: 1. Fedez distance: 3.12 Fabri Fibra distance: 3.12
Shapelet number: 2. Fedez distance: 4.96 Fabri Fibra distance: 4.83
Shapelet number: 3. Fedez distance: 5.30 Fabri Fibra distance: 5.18
